# 05 · Seleção manual e sync Kaggle Dataset → SSD

Escolha somente os arquivos que quer carregar. O sync não baixa o Dataset inteiro: ele usa o filtro `-f` do Kaggle CLI para transferir apenas os arquivos selecionados.


In [ ]:
import sys
from pathlib import Path
import subprocess

SCRIPTS_DIR = Path("/kaggle/working/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))
DATASET = "automamermaid/comfydocs"
TARGET_DIR = Path("/kaggle/working/ComfyUI/models")

from kaggle_sync import get_dataset_files, get_dataset_files_details, filter_dataset_files, sync_dataset_to_local, MODEL_CATEGORIES

def choose_dataset_files(dataset=DATASET, preselected_categories=None):
    """Abre uma caixa de seleção mostrando nome, categoria/path e tamanho formatado."""
    try:
        details = get_dataset_files_details(dataset)
    except Exception as e:
        print(f"[WARN] Falha ao obter detalhes: {e}")
        details = []

    if details:
        if preselected_categories:
            candidates = [d for d in details if d["category"] in preselected_categories]
        else:
            candidates = details
        if not candidates:
            candidates = details
        options = [(f"{d['name']}  [{d['category']}/]  ({d['size']})", d["path"]) for d in candidates]
        paths = [d["path"] for d in candidates]
    else:
        raw_files = get_dataset_files(dataset)
        if not raw_files:
            raise RuntimeError("O Dataset não possui arquivos ou não pôde ser listado.")
        filtered = filter_dataset_files(raw_files, categories=preselected_categories)
        candidates_files = filtered if filtered else raw_files
        options = [(f"{Path(f).name}  [{f}]", f) for f in candidates_files]
        paths = candidates_files

    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        selector = widgets.SelectMultiple(
            options=options,
            rows=min(18, max(5, len(options))),
            description="Modelos:",
            layout=widgets.Layout(width="100%", height="420px"),
        )
        select_all = widgets.Button(description="Selecionar todos")
        clear_all = widgets.Button(description="Limpar")
        confirm = widgets.Button(description="Confirmar seleção", button_style="success")
        output = widgets.Output()

        def all_click(_): selector.value = tuple(paths)
        def clear_click(_): selector.value = tuple()
        def confirm_click(_):
            with output:
                clear_output(wait=True)
                chosen = list(selector.value)
                if not chosen:
                    print("Nenhum arquivo selecionado.")
                else:
                    print("Selecionados:")
                    for item in chosen: print("  -", item)
                    print(f"Total: {len(chosen)} arquivo(s)")

        select_all.on_click(all_click)
        clear_all.on_click(clear_click)
        confirm.on_click(confirm_click)
        display(widgets.HTML(f"<b>{dataset}</b> · {len(options)} arquivo(s) disponível(is)"))
        display(widgets.HBox([select_all, clear_all, confirm]))
        display(selector, output)

        return selector
    except ImportError:
        print("ipywidgets não disponível. Use a lista abaixo e informe os paths manualmente.")
        for i, opt in enumerate(options, 1):
            print(f"{i:03d}: {opt[0]}")
        raw = input("Números separados por vírgula: ").strip()
        indexes = [int(x)-1 for x in raw.split(',') if x.strip().isdigit()]
        return [paths[i] for i in indexes if 0 <= i < len(paths)]


In [ ]:
# Filtre opcional por categoria antes de abrir a caixa de seleção.
CATEGORIES = None  # Ex.: ["checkpoints", "loras"]
selector = choose_dataset_files(DATASET, preselected_categories=CATEGORIES)


In [ ]:
# Depois de marcar os arquivos na caixa acima, execute esta célula.
# `selector` é o widget criado na célula anterior.
if hasattr(selector, "value"):
    SELECTED_FILES = list(selector.value)
else:
    SELECTED_FILES = list(selector)

if not SELECTED_FILES:
    raise ValueError("Nenhum arquivo selecionado. Volte à célula anterior e escolha pelo menos um.")

stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=TARGET_DIR,
    selected_files=SELECTED_FILES,
    force=False,
)

print("\nSYNC CONCLUÍDO")
print(f"Sincronizados: {stats['synced']}")
print(f"Pulados: {stats['skipped']}")
print(f"Erros: {stats['errors']}")
